In [ ]:
from pathlib import Path
import json

from PIL import Image
import matplotlib.pyplot as plt

RUN_DIR = Path("outputs/hero_sam3")  # change to whichever run you want

In [ ]:
manifest_path = RUN_DIR / "conceptops_manifest.json"
events_path = RUN_DIR / "events.json"
concepts_path = RUN_DIR / "concepts.json"

with manifest_path.open() as f:
    manifest = json.load(f)

with events_path.open() as f:
    events_payload = json.load(f)

with concepts_path.open() as f:
    concepts_payload = json.load(f)

events = events_payload.get("events", [])
events_concepts = concepts_payload.get("events_concepts", [])

len(events), len(events_concepts), manifest["stages"]

In [ ]:
# Build lookup: event_id -> concept entry
concept_by_event = {c["event_id"]: c for c in events_concepts}

for ev in events:
    c = concept_by_event.get(ev["event_id"])
    if not c:
        continue
    print(
        f"Event {ev['event_id']}: frames {ev['start_frame']}–{ev['end_frame']}, "
        f"labels={c['labels']}, scores={ [round(s, 2) for s in c['scores']] }"
    )

In [ ]:
thumbs_dir = RUN_DIR / "thumbnails"

fig_cols = 4
fig_rows = max(1, (len(events_concepts) + fig_cols - 1) // fig_cols)
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(4 * fig_cols, 4 * fig_rows))
axes = axes.flatten()

for ax in axes[len(events_concepts):]:
    ax.axis("off")

for ax, c in zip(axes, events_concepts):
    thumb_path = Path(c["thumbnail_path"])
    img = Image.open(thumb_path).convert("RGB")
    ax.imshow(img)
    labels = c["labels"]
    scores = [round(s, 2) for s in c["scores"]]
    title = f"Event {c['event_id']}\n" + ", ".join(
        f"{lbl}({sc})" for lbl, sc in zip(labels, scores)
    )
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# choose which event to inspect
EVENT_ID = 0

ev = next(e for e in events if e["event_id"] == EVENT_ID)
print(ev)

frames_dir = Path(manifest["frames_dir"])

frame_paths = []
for idx in range(ev["start_frame"], ev["end_frame"] + 1):
    # Assumes frame filenames like frame_000000.jpg
    frame_paths.append(frames_dir / f"frame_{idx+1:06d}.jpg")

len(frame_paths), frame_paths[:3]

In [ ]:
# choose which event to inspect
EVENT_ID = 0

ev = next(e for e in events if e["event_id"] == EVENT_ID)
print(ev)

frames_dir = Path(manifest["frames_dir"])

frame_paths = []
for idx in range(ev["start_frame"], ev["end_frame"] + 1):
    # Assumes frame filenames like frame_000000.jpg
    frame_paths.append(frames_dir / f"frame_{idx+1:06d}.jpg")

len(frame_paths), frame_paths[:3]

In [ ]:
n = len(frame_paths)
cols = min(6, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = axes.flatten()

for ax in axes[n:]:
    ax.axis("off")

for ax, p in zip(axes, frame_paths):
    if not p.exists():
        ax.axis("off")
        continue
    img = Image.open(p).convert("RGB")
    ax.imshow(img)
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()